In [1]:
from pathlib import Path

import pandas as pd
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

c:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RANDOM_SEED = 42

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
models_dir = project_root / "models"

models_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(processed_dir / "train.csv")

print("Full training set:", train_df.shape)

# Use only 100 examples for this technical smoke test.
smoke_train_df = train_df.sample(
    n=100,
    random_state=RANDOM_SEED
).copy()

print("Smoke-test training set:", smoke_train_df.shape)

Full training set: (3385, 3)
Smoke-test training set: (100, 3)


In [3]:
smoke_train_dataset = Dataset.from_pandas(
    smoke_train_df[
        ["sentence", "label_id"]
    ].rename(columns={"label_id": "labels"}),
    preserve_index=False
)


In [4]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=256
    )

tokenized_smoke_train = smoke_train_dataset.map(
    tokenize_function,
    batched=True
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

c:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sarth\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 100/100 [00:00<00:00, 1356.45 e

In [5]:
id_to_label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label_to_id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

smoke_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id_to_label,
    label2id=label_to_id
)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2587.10it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
smoke_training_args = TrainingArguments(
    output_dir=str(models_dir / "smoke_checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=RANDOM_SEED
)

smoke_trainer = Trainer(
    model=smoke_model,
    args=smoke_training_args,
    train_dataset=tokenized_smoke_train,
    data_collator=data_collator,
    processing_class=tokenizer
)

smoke_trainer.train()

print("Smoke test completed successfully.")

c:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,1.092774
10,1.072276


Smoke test completed successfully.
